Name: Christian Aron Vea

GitHub: https://github.com/ChristianAronVea/IND320-Vea

Streamlit: https://ind320-vea-5sc8vxsjg7v4bjgnpamvfl.streamlit.app/

Recording: 

AI usage and log can be found in separate files in the repo.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

In [2]:
import geopandas as gpd

K = 1000
region_coordinates = {}

# Electricity areas. The budomr column contains values such as "NO 3".
gdf = gpd.read_file("data/Budområder/Budområder.shp")
points_per_region = gdf["geometry"].sample_points(size=K, random_state=42)
points_wgs84 = points_per_region.to_crs(epsg=4326)

for idx, multipoint in points_wgs84.items():
    region_id = gdf.loc[idx, "budomr"].replace(" ", "")
    region_coordinates[region_id] = []

    for point in multipoint.geoms:
        lon = point.x
        lat = point.y
        region_coordinates[region_id].append((lon, lat))        # Store coordinates as (lon, lat) pairs for the API.

# Precipitation areas.
gdf = gpd.read_file("data/NVE_Region/NVE_Region.shp", encoding="UTF-8")
points_per_region = gdf["geometry"].sample_points(size=K, random_state=42)
points_wgs84 = points_per_region.to_crs(epsg=4326)

for idx, multipoint in points_wgs84.items():
    region_id = gdf.loc[idx, "REGION"]
    region_coordinates[region_id] = []

    for point in multipoint.geoms:
        lon = point.x
        lat = point.y
        region_coordinates[region_id].append((lon, lat))  # Store coordinates as (lon, lat) pairs for the API.

# Coordinates are stored as (longitude, latitude) pairs for the API.
{region: len(points) for region, points in region_coordinates.items()}

{'NO1': 1000,
 'NO2': 1000,
 'NO3': 1000,
 'NO4': 1000,
 'NO5': 1000,
 'RV': 1000,
 'RM': 1000,
 'RN': 1000,
 'RØ': 1000,
 'RS': 1000}

In [3]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [4]:
from pathlib import Path

url = "https://archive-api.open-meteo.com/v1/archive"

def get_weather_data(region, start_date, end_date):
    data_folder = Path("data/weather_batches")
    data_folder.mkdir(parents=True, exist_ok=True)

    coordinates = region_coordinates[region]
    batch_size = 25
    batch_files = []

    for batch_start in range(0, len(coordinates), batch_size):
        batch = coordinates[batch_start:batch_start + batch_size]

        file_name = (
            f"{region}_{start_date}_{end_date}_"
            f"batch_{batch_start:04d}.csv.gz"
        )
        batch_file = data_folder / file_name
        batch_files.append(batch_file)

        # Hopp over batcher som allerede er lagret.
        if batch_file.exists():
            print(f"Batch {batch_start // batch_size + 1} finnes allerede; hopper over.")
            continue

        longitudes, latitudes = zip(*batch)

        params = {
            "latitude": list(latitudes),
            "longitude": list(longitudes),
            "start_date": start_date,
            "end_date": end_date,
            "hourly": ["temperature_2m", "precipitation"],
            "models": "era5",
            "timezone": "UTC",
        }

        responses = openmeteo.weather_api(url, params=params)

        if len(responses) != len(batch):
            raise ValueError(
                f"Forventet {len(batch)} svar, men fikk {len(responses)}."
            )

        batch_data = []

        for point_id, (response, (lon, lat)) in enumerate(
            zip(responses, batch),
            start=batch_start,
        ):
            hourly = response.Hourly()

            point_data = pd.DataFrame({
                "date": pd.date_range(
                    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                    freq=pd.Timedelta(seconds=hourly.Interval()),
                    inclusive="left",
                ),
                "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
                "precipitation": hourly.Variables(1).ValuesAsNumpy(),
            })

            point_data["point_id"] = point_id
            point_data["longitude"] = lon
            point_data["latitude"] = lat
            batch_data.append(point_data)

        batch_dataframe = pd.concat(batch_data, ignore_index=True)

        # Skriv først til en midlertidig fil, og gi den endelig navn når lagringen er ferdig.
        temporary_file = batch_file.with_name(batch_file.name + ".tmp")
        batch_dataframe.to_csv(
            temporary_file,
            index=False,
            compression="gzip",
        )
        temporary_file.replace(batch_file)

        print(
            f"Lagret batch {batch_start // batch_size + 1}/"
            f"{(len(coordinates) + batch_size - 1) // batch_size}"
        )

    data = [
        pd.read_csv(batch_file, parse_dates=["date"])
        for batch_file in batch_files
    ]
    return pd.concat(data, ignore_index=True)

In [ ]:
NO3_2024_data = get_weather_data("NO3", "2024-01-01", "2024-12-31")

In [ ]:
NO3_2024_data.to_parquet("data/NO3_2024.parquet", index=False)       # save the data to a parquet file for future use
#NO3_2024_data = pd.read_parquet("data/NO3_2024.parquet")             # read the data from the parquet file if needed

NO3_2024_data

,date,temperature_2m,precipitation,point_id,longitude,latitude
0,2024-01-01 00:00:00+00:00,-8.997499,0.0,0,8.116789,61.688542
1,2024-01-01 01:00:00+00:00,-9.247499,0.0,0,8.116789,61.688542
2,2024-01-01 02:00:00+00:00,-9.447499,0.0,0,8.116789,61.688542
3,2024-01-01 03:00:00+00:00,-9.597500,0.0,0,8.116789,61.688542
4,2024-01-01 04:00:00+00:00,-9.697499,0.0,0,8.116789,61.688542
...,...,...,...,...,...,...
47995,2024-01-02 19:00:00+00:00,-5.000000,0.0,999,10.141907,64.545548
47996,2024-01-02 20:00:00+00:00,-4.950000,0.0,999,10.141907,64.545548
47997,2024-01-02 21:00:00+00:00,-4.950000,0.0,999,10.141907,64.545548
47998,2024-01-02 22:00:00+00:00,-5.000000,0.0,999,10.141907,64.545548


## Outliers and anomalies

The precipitation limits are calculated from the DCT high-pass residual, but the plot shows the original hourly regional mean. The plotted limits are the slowly varying baseline plus robust limits; the residual itself is not plotted.

In [ ]:
from scipy.fft import dct, idct
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler


def high_pass_dct(values, cutoff_period_days=30):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or len(values) < 3:
        raise ValueError("values must be a one-dimensional series with at least 3 values.")
    if not np.isfinite(values).all():
        raise ValueError("values must not contain missing or infinite values.")
    if cutoff_period_days <= 0:
        raise ValueError("cutoff_period_days must be greater than zero.")

    coefficients = dct(values, type=2, norm="ortho")
    low_frequency_count = int(np.ceil(2 * len(values) / (cutoff_period_days * 24)))
    coefficients[:max(1, low_frequency_count)] = 0
    return idct(coefficients, type=2, norm="ortho")


def get_hourly_series(weather_data, variable):
    if "time" not in weather_data or variable not in weather_data:
        raise KeyError(f"weather_data must contain 'time' and {variable!r} columns.")

    data = weather_data[["time", variable]].copy()
    data["time"] = pd.to_datetime(data["time"], utc=True)
    data = data.sort_values("time").set_index("time")
    if data.index.has_duplicates:
        raise ValueError("weather_data contains duplicate timestamps.")
    if data[variable].isna().any():
        raise ValueError(f"{variable} contains missing values.")
    if len(data) > 1 and not (data.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all():
        raise ValueError("weather_data must have a regular hourly time interval.")
    return data[variable].astype(float)


def robust_scale(values):
    center = np.median(values)
    scale = 1.4826 * np.median(np.abs(values - center))
    if scale == 0:
        scale = (np.quantile(values, 0.75) - np.quantile(values, 0.25)) / 1.349
    if scale == 0:
        scale = np.std(values)
    return center, scale


def plot_precipitation_outliers(weather_data, cutoff_period_days=30, n_std=3):
    if n_std <= 0:
        raise ValueError("n_std must be greater than zero.")

    precipitation = get_hourly_series(weather_data, "precipitation")
    satv = high_pass_dct(precipitation.to_numpy(), cutoff_period_days)
    center, scale = robust_scale(satv)
    baseline = precipitation.to_numpy() - satv
    lower_limit = baseline + center - n_std * scale
    upper_limit = baseline + center + n_std * scale
    outlier_mask = (satv < center - n_std * scale) | (satv > center + n_std * scale)

    fig, axis = plt.subplots(figsize=(15, 5))
    axis.plot(precipitation.index, precipitation, color="steelblue", linewidth=0.8,
              label="Mean precipitation")
    axis.plot(precipitation.index, upper_limit, color="darkorange", linewidth=1.2,
              label=f"Robust limits ({n_std} sigma)")
    axis.plot(precipitation.index, lower_limit, color="darkorange", linewidth=1.2)
    axis.scatter(precipitation.index[outlier_mask], precipitation.to_numpy()[outlier_mask],
                 color="crimson", s=18, label="Outlier", zorder=3)
    axis.set(title="Hourly mean precipitation in NO3 with SPC outliers",
             xlabel="Time (UTC)", ylabel="Precipitation (mm)")
    axis.grid(True, alpha=0.3)
    axis.legend()
    fig.tight_layout()

    outliers = pd.DataFrame({
        "time": precipitation.index[outlier_mask],
        "precipitation": precipitation.to_numpy()[outlier_mask],
        "adjusted_residual": satv[outlier_mask],
        "lower_limit": lower_limit[outlier_mask],
        "upper_limit": upper_limit[outlier_mask],
    })
    return fig, outliers


def plot_temperature_lof(weather_data, proportion_outliers=0.01,
                         cutoff_period_days=30, n_neighbors=20):
    if not 0 < proportion_outliers < 0.5:
        raise ValueError("proportion_outliers must be between 0 and 0.5.")

    temperature = get_hourly_series(weather_data, "temperature_2m")
    if len(temperature) < 3:
        raise ValueError("At least 3 hourly observations are required for LOF.")
    residual = high_pass_dct(temperature.to_numpy(), cutoff_period_days)
    hourly_change = np.diff(residual, prepend=residual[0])
    features = StandardScaler().fit_transform(np.column_stack((residual, hourly_change)))
    model = LocalOutlierFactor(
        n_neighbors=min(n_neighbors, len(temperature) - 1),
        contamination=proportion_outliers,
    )
    is_outlier = model.fit_predict(features) == -1
    lof_score = -model.negative_outlier_factor_

    fig, axis = plt.subplots(figsize=(15, 5))
    axis.plot(temperature.index, temperature, color="darkgreen", linewidth=0.8,
              label="Mean temperature")
    axis.scatter(temperature.index[is_outlier], temperature.to_numpy()[is_outlier],
                 color="crimson", s=20, label="LOF anomaly", zorder=3)
    axis.set(title="Hourly mean temperature in NO3 with LOF anomalies",
             xlabel="Time (UTC)", ylabel="Temperature (°C)")
    axis.grid(True, alpha=0.3)
    axis.legend()
    fig.tight_layout()

    outliers = pd.DataFrame({
        "time": temperature.index[is_outlier],
        "temperature_2m": temperature.to_numpy()[is_outlier],
        "lof_score": lof_score[is_outlier],
    })
    return fig, outliers

In [ ]:
test_time = pd.date_range("2024-01-01", periods=24 * 30, freq="h", tz="UTC")
test_hours = np.arange(len(test_time))
test_precipitation = 0.5 + 0.3 * np.sin(2 * np.pi * test_hours / 24)
test_precipitation[301] += 15
test_temperature = 8 + 0.03 * test_hours + 4 * np.sin(2 * np.pi * test_hours / 24)
test_temperature[401] += 20
test_weather = pd.DataFrame({
    "time": test_time,
    "precipitation": test_precipitation,
    "temperature_2m": test_temperature,
})

test_precipitation_fig, test_precipitation_outliers = plot_precipitation_outliers(
    test_weather, cutoff_period_days=7, n_std=2.5
)
assert test_time[301] in pd.DatetimeIndex(test_precipitation_outliers["time"])
plt.close(test_precipitation_fig)

test_temperature_fig, test_temperature_outliers = plot_temperature_lof(
    test_weather, proportion_outliers=0.01, cutoff_period_days=7
)
assert test_time[401] in pd.DatetimeIndex(test_temperature_outliers["time"])
plt.close(test_temperature_fig)

precipitation_fig, precipitation_outliers = plot_precipitation_outliers(weather_no3_2024)
temperature_fig, temperature_outliers = plot_temperature_lof(weather_no3_2024)
plt.show()
print(f"Precipitation outliers: {len(precipitation_outliers)}")
print(f"Temperature anomalies: {len(temperature_outliers)}")
display(precipitation_outliers.head(), temperature_outliers.head())

## Reservoir filling: STL and spectrogram

The NVE Magasinstatistikk API uses `omrType="EL"` and `omrnr=3` for electricity area NO3. This weekly series is used for both plots.

In [ ]:
MAGASIN_API_URL = (
    "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"
)
response = requests.get(MAGASIN_API_URL, timeout=60)
response.raise_for_status()
reservoir_data = pd.DataFrame(response.json())
reservoir_data["dato_Id"] = pd.to_datetime(reservoir_data["dato_Id"])
reservoir_data["omrnr"] = pd.to_numeric(reservoir_data["omrnr"])

no3_fill = (
    reservoir_data.loc[
        (reservoir_data["omrType"] == "EL") & (reservoir_data["omrnr"] == 3),
        ["dato_Id", "fyllingsgrad"],
    ]
    .drop_duplicates("dato_Id")
    .set_index("dato_Id")["fyllingsgrad"]
    .sort_index()
    .astype(float)
)
no3_fill.name = "fill_rate"
if no3_fill.empty:
    raise ValueError("The Magasinstatistikk API returned no filling data for NO3.")
no3_fill.head()

In [ ]:
from scipy.signal import spectrogram
from statsmodels.tsa.seasonal import STL


def plot_stl_decomposition(fill_rate, seasonal=13, trend=53, robust=True):
    series = fill_rate.dropna().astype(float).sort_index()
    if seasonal < 7 or seasonal % 2 == 0:
        raise ValueError("seasonal must be an odd integer of at least 7.")
    if trend <= 52 or trend % 2 == 0:
        raise ValueError("trend must be an odd integer greater than the 52-week period.")
    if len(series) < 2 * 52:
        raise ValueError("At least two years of weekly fill-rate data are required for STL.")
    if series.index.has_duplicates:
        raise ValueError("fill_rate contains duplicate dates.")

    result = STL(
        series,
        period=52,
        seasonal=seasonal,
        trend=trend,
        robust=robust,
    ).fit()
    fig = result.plot()
    fig.set_size_inches(14, 9)
    fig.suptitle("NO3 reservoir filling: STL decomposition", y=1.01)
    fig.tight_layout()
    return fig


def plot_fill_rate_spectrogram(fill_rate, window_length=52, window_overlap=26):
    series = fill_rate.dropna().astype(float).sort_index()
    if window_length < 2 or window_length > len(series):
        raise ValueError("window_length must be between 2 and the series length.")
    if window_overlap < 0 or window_overlap >= window_length:
        raise ValueError("window_overlap must be non-negative and less than window_length.")

    frequencies, times, power = spectrogram(
        series.to_numpy(),
        fs=1.0,
        window="hann",
        nperseg=window_length,
        noverlap=window_overlap,
        scaling="density",
        mode="psd",
    )
    dates = series.index[0] + pd.to_timedelta(times * 7, unit="D")
    power_db = 10 * np.log10(np.maximum(power[1:], np.finfo(float).tiny))

    fig, axis = plt.subplots(figsize=(14, 6))
    image = axis.pcolormesh(
        dates,
        frequencies[1:],
        power_db,
        shading="auto",
        cmap="viridis",
    )
    axis.set_yscale("log")
    axis.set(title="NO3 reservoir filling: spectrogram",
             xlabel="Time", ylabel="Frequency (cycles per week)")
    fig.colorbar(image, ax=axis, label="Power (dB)")
    fig.tight_layout()
    return fig

In [ ]:
test_weekly_dates = pd.date_range("2020-01-05", periods=5 * 52, freq="W-SUN")
test_fill = pd.Series(
    0.5 + 0.1 * np.sin(2 * np.pi * np.arange(len(test_weekly_dates)) / 52),
    index=test_weekly_dates,
)
test_stl_fig = plot_stl_decomposition(test_fill)
test_spectrogram_fig = plot_fill_rate_spectrogram(test_fill)
assert len(test_stl_fig.axes) >= 4
assert len(test_spectrogram_fig.axes) >= 2
plt.close(test_stl_fig)
plt.close(test_spectrogram_fig)

stl_fig = plot_stl_decomposition(no3_fill, seasonal=13, trend=53, robust=True)
spectrogram_fig = plot_fill_rate_spectrogram(
    no3_fill, window_length=52, window_overlap=26
)
plt.show()